# Node Classification on Cora Benchmark

Node Classification on Cora (Planetoid): Benchmark comparison for semi-supervised node classification on Cora. This notebook implements the approach with `SplineConv / GCNConv` inside a `K3Net` model, trained with the Adam optimizer for 100 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SplineConv / GCNConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "Node Classification on Cora Benchmark (SplineConv / GCNConv)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.Compose([k3_transforms.TargetIndegree()]))
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. SplineConv Model with GCNConv Fallback
class K3Net(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        try:
            self.conv1 = k3_layers.SplineConv(in_channels, hidden_channels, dim=1, kernel_size=2)
            self.conv2 = k3_layers.SplineConv(hidden_channels, out_channels, dim=1, kernel_size=2)
            self.use_spline = True
        except Exception:
            self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
            self.conv2 = k3_layers.GCNConv(hidden_channels, out_channels)
            self.use_spline = False
        self.dropout = layers.Dropout(0.5)

    def call(self, inputs, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
            edge_attr = inputs[2] if len(inputs) > 2 else None
        else:
            x, edge_index, edge_attr = inputs, None, None

        x = self.dropout(x, training=training)
        if self.use_spline and edge_attr is not None:
            x = ops.elu(self.conv1(x, edge_index, edge_attr=edge_attr))
            x = self.dropout(x, training=training)
            return self.conv2(x, edge_index, edge_attr=edge_attr)
        else:
            x = ops.relu(self.conv1(x, edge_index))
            x = self.dropout(x, training=training)
            return self.conv2(x, edge_index)

k3_model = K3Net(num_features, 16, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=5e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Training
def graph_data_generator():
    x = ops.convert_to_tensor(data.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data.edge_index, dtype="int64")
    y = ops.convert_to_tensor(data.y, dtype="int64")
    mask = ops.cast(data.train_mask, "float32")
    edge_attr = ops.convert_to_tensor(data.edge_attr, dtype="float32")
    while True:
        yield (x, edge_index, edge_attr), y, mask

print(f"Training K3-Node Cora model on {backend} backend...")
history = k3_model.fit(
    graph_data_generator(),
    steps_per_epoch=1,
    epochs=100,
    verbose=1,
)

# 5. Evaluation
out = k3_model((data.x, data.edge_index, data.edge_attr))
pred = ops.argmax(out, axis=-1)
test_mask = data.test_mask
test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data.y[test_mask], "int64"), "float32"))
print(f"Test Accuracy: {float(test_acc):.4f}")

print("\n✓ K3-Node execution completed successfully!")